In [85]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

In [86]:
import deepxde as dde
#from deepxde.backend import tf
dde.backend.backend_name = "pytorch"

import torch
import matplotlib.pyplot as plt
import numpy as np

In [87]:
print(os.environ["DDE_BACKEND"])

pytorch


In [72]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x)
    dy_xxxx = dde.grad.hessian(dy_xx, x)
    return dy_xxxx + 1

In [73]:
geom = dde.geometry.Interval(0, 1)

In [74]:
def boundary(x, on_boundary):
    return on_boundary and dde.utils.isclose(x[0], 0)

def boundary_value(X):
    return 0



In [75]:
#Custom boundary condition

def ddy(x, y):
    return dde.grad.hessian(y, x)

def dddy(x, y):
    return dde.grad.jacobian(ddy(x, y), x)

def boundary_second_derivative(x, y, _):
    return ddy(x, y)

def boundary_triple_derivative(x, y, _): #'_' is a placeholder value for syntax wise required varibales but the ones which arent being used
    return dddy(x, y)

def boundary_r(X, on_boundary):
    return on_boundary and dde.utils.isclose(X[0], 1)

In [76]:
#Exact solution 
def exact_sol(x):
    return - (x**4) / 24 + (x**3)/6 - (x**2)/4

In [77]:
#Left boundaries with neumann and dirichlet
bc_dbc = dde.icbc.DirichletBC(geom, boundary_value, boundary)
bc_nbc = dde.icbc.NeumannBC(geom, boundary_value, boundary)

In [78]:
bc_obc1 = dde.icbc.OperatorBC(geom, boundary_second_derivative, boundary_r)
bc_obc2 = dde.icbc.OperatorBC(geom, boundary_triple_derivative, boundary_r)

In [79]:
data = dde.data.PDE(geom, pde, [bc_dbc, bc_nbc, bc_obc1, bc_obc2], 32, 2, solution=exact_sol, num_test=100)

In [80]:
layer_size = [1] + [30]*5 + [1]
activation = "tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)

model = dde.Model(data, net)
model.compile("adam", lr=0.001, metrics=["l2 relative error"])
losshistory, train_state = model.train(3000)

Compiling model...
'compile' took 0.002056 s

Training model...



/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


Step      Train loss                                            Test loss                                             Test metric   
0         [4.13e+00, 0.00e+00, 5.67e-02, 1.85e-02, 7.92e-02]    [4.36e+00, 0.00e+00, 5.67e-02, 1.85e-02, 7.92e-02]    [2.67e+00]    
1000      [8.34e-04, 2.48e-07, 2.55e-06, 2.54e-05, 6.89e-07]    [6.60e-04, 2.48e-07, 2.55e-06, 2.54e-05, 6.89e-07]    [3.77e-02]    
2000      [1.39e-05, 1.59e-11, 2.29e-09, 1.15e-08, 5.33e-09]    [1.09e-05, 1.59e-11, 2.29e-09, 1.15e-08, 5.33e-09]    [2.16e-04]    
3000      [7.58e-06, 6.12e-11, 5.56e-10, 6.20e-09, 1.82e-09]    [5.78e-06, 6.12e-11, 5.56e-10, 6.20e-09, 1.82e-09]    [1.71e-04]    

Best model at step 3000:
  train loss: 7.59e-06
  test loss: 5.79e-06
  test metric: [1.71e-04]

'train' took 8.194428 s



MODEL.NET() :- 
1. gives direct access to the underlying neural network
2. allows forward pass, layer access etc.
3. requires backend specific tensors as inputs, not numpy arrays

In [81]:
input_tensor = torch.tensor([[0.1], [0.2], [0.3]], dtype=torch.float32, requires_grad=True)

outputs = model.net(input_tensor)

print(outputs)

#Computing gradients
outputs.sum().backward()
print("Gradients: ", input_tensor.grad)



AttributeError: Exception encountered when calling layer 'fnn_4' (type FNN).

'torch.Size' object has no attribute 'rank'

Call arguments received by layer 'fnn_4' (type FNN):
  • inputs=tensor([[0.1000],
        [0.2000],
        [0.3000]], requires_grad=True)
  • training=False

ERROR METHODS 
f(X, inputs, outputs, beg, end) where are beg and end are the first and the last indices.

In [82]:
#For DIRICHLET
X_boundary = np.array([[0.0]])

inputs = torch.tensor(X_boundary, dtype=torch.float32)

outputs = torch.tensor(model.predict(X_boundary), dtype=torch.float32)

beg, end = 0, len(X_boundary)

residual_left = bc_dbc.error(X_boundary, inputs, outputs, beg, end)
mse_residual = np.mean(residual_left**2)

print(mse_residual.item())


6.190248313941993e-11


In [83]:
#FOR NEUMANN(need to use backend specific input tensor)
X_test = torch.tensor([[0.0]], dtype=torch.float32, requires_grad=True)

net = model.net
outputs = net(X_test)

beg, end = 0, len(X_test)

residual_nbc = bc_nbc.error(X_test.cpu().detach().numpy(),
                            X_test,
                            outputs,
                            beg,
                            end)

mse_residual_nbc = torch.mean(residual_nbc**2)

print(mse_residual_nbc.item())

AttributeError: Exception encountered when calling layer 'fnn_4' (type FNN).

'torch.Size' object has no attribute 'rank'

Call arguments received by layer 'fnn_4' (type FNN):
  • inputs=tensor([[0.]], requires_grad=True)
  • training=False

In [84]:
#FOR OPERATORBC
X_test = torch.tensor([[1.0]], dtype=torch.float32, requires_grad=True).to("cuda")

net = model.net
outputs = net(X_test)

beg, end = 0, len(X_test)

residual_obc1 = bc_obc1.error(
    X_test.cpu().detach().numpy(),
    X_test,
    outputs,
    beg, 
    end
)

residual_obc2 = bc_obc2.error(
    X_test.cpu().detach().numpy(),
    X_test,
    outputs,
    beg,
    end
)

mse_residual_obc1 = torch.mean(residual_obc1**2)
mse_residual_obc2 = torch.mean(residual_obc2**2)

print(f"OBC1: {mse_residual_obc1}, OBC2: {mse_residual_obc2}")

AttributeError: Exception encountered when calling layer 'fnn_4' (type FNN).

'torch.Size' object has no attribute 'rank'

Call arguments received by layer 'fnn_4' (type FNN):
  • inputs=tensor([[1.]], device='cuda:0', grad_fn=<ToCopyBackward0>)
  • training=False